# 02 — Annotation extraction: Kimi Direct

This notebook runs P0–P6 over the human validation set through Moonshot's
direct Kimi K3 API. Each prompt section repeats the configuration and execution
pattern from cells 3 and 4 of the original Kimi Direct notebook.

Change `N_TEST_DIALOGUES` inside any section to control that prompt
independently. Each section extracts and scores the first N validation
dialogues in stable dialogue-ID order.

The dialogue records use the logical split `validation`. The shared cache
helper maps this split to `train`, so records are stored under:

```text
extension/artifacts/extraction_cache/train/moonshot-direct__kimi-k3-max/{prompt}/{dialogue_id}.json
```

P0–P5 use codebooks v0–v5 respectively. P6 is the same experiment structure
and prompt format, using codebook v6.


## 1. Load the validation data

The dataframe retains human annotations for scoring. Only the conversation and
ordered unit names are passed to prompt construction.


In [ ]:
import os, sys
from pathlib import Path

_here = Path.cwd()
for _candidate in [_here, *_here.parents]:
    if (_candidate / 'extension' / 'artifacts').exists():
        os.chdir(_candidate)
        break
else:
    raise FileNotFoundError('Run this notebook from inside the Experiment_1 repository.')

sys.path.insert(0, str(Path.cwd()))

from extension.scripts.load_annotation_data import load_dataset
from extension.scripts import prompt_loader, extraction, scoring
from extension.scripts.model_specific import moonshot_kimi

try:
    moonshot_kimi._api_key()
    _key = True
except RuntimeError:
    _key = False

print('cwd:', os.getcwd(), '| MOONSHOT key found:', _key)


In [ ]:
VALIDATION_PATH = 'extension/artifacts/annotation_dev_and_val_sets/validation_set.csv'
SPLIT = 'validation'

gold = load_dataset(VALIDATION_PATH)
DIALOGUES = extraction.dialogues_from(gold, split=SPLIT)

assert all(dialogue['split'] == SPLIT for dialogue in DIALOGUES)
print(f'{len(DIALOGUES)} validation dialogues, {len(gold)} units')
print('validation cache maps to:',
      extraction.cache_path(
          moonshot_kimi.cache_slug('max'), 'P0',
          DIALOGUES[0]['dialogue_id'], SPLIT,
      ).parent)


## 2. Extract annotations with P0: codebook v0


In [ ]:
TEST_PROMPTS = ['P0']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


## 3. Extract annotations with P1: codebook v1


In [ ]:
TEST_PROMPTS = ['P1']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


## 4. Extract annotations with P2: codebook v2


In [ ]:
TEST_PROMPTS = ['P2']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


## 5. Extract annotations with P3: codebook v3


In [ ]:
TEST_PROMPTS = ['P3']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


## 6. Extract annotations with P4: codebook v4


In [ ]:
TEST_PROMPTS = ['P4']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


## 7. Extract annotations with P5: codebook v5


In [ ]:
TEST_PROMPTS = ['P5']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))


## 8. Extract annotations with P6: codebook v6


In [ ]:
TEST_PROMPTS = ['P6']
N_TEST_DIALOGUES = 78
MAX_WORKERS = 10
EFFORT = 'max'        # K3 scale: 'low', 'high', 'max' (platform default)


In [ ]:
TEST_DIALOGUES = DIALOGUES[:N_TEST_DIALOGUES]
for _p in TEST_PROMPTS:
    assert _p in prompt_loader.list_prompts(), f"unknown prompt {_p!r}"
SLUG = moonshot_kimi.cache_slug(EFFORT)
print(f"moonshot-direct kimi-k3 effort={EFFORT} on "
      f"{[d['dialogue_id'] for d in TEST_DIALOGUES]} x {TEST_PROMPTS}\n")

import json as _json
import pandas as pd
family_f1_cols = [f'f1_{family}' for family in scoring.FAMILIES]
pooled_family_f1_cols = [f'pooled_f1_{family}' for family in scoring.POOLED_FAMILIES]
test_rows = []
for prompt in TEST_PROMPTS:
    print(f"  {prompt} ({len(TEST_DIALOGUES)} dialogues, {MAX_WORKERS} workers):")
    moonshot_kimi.generate_annotations(prompt, TEST_DIALOGUES,
                                       reasoning_effort=EFFORT, max_workers=MAX_WORKERS)
    for dlg in TEST_DIALOGUES:
        rec = _json.load(open(extraction.cache_path(SLUG, prompt,
                                                    dlg['dialogue_id'], dlg['split'])))
        u = (rec['attempts'][0].get('meta') or {}).get('usage', {}) if rec['attempts'] else {}
        print(f"  {prompt:22s} {dlg['dialogue_id']}: {'ok' if rec['valid'] else 'invalid':8s} "
              f"completion {u.get('completion_tokens','?')} tok  latency {rec['latency_s']:.1f}s")
    s = scoring.score_config(gold, SLUG, prompt,
                             [d['dialogue_id'] for d in TEST_DIALOGUES], n_boot=0,
                             split=SPLIT)
    test_rows.append(s)
    print(f"  -> validity {s['valid_rate']:.0%} | macro-F1(P) {s['macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['weighted_f1_P']:.3f} "
          f"| accuracy {s['accuracy']:.3f} | alpha {s['alpha']:.3f}")
    print("     family F1(P): " + " | ".join(
        f"{family}={s[f'f1_{family}']:.3f}" for family in scoring.FAMILIES
    ))
    print(f"     pooled: macro-F1(P) {s['pooled_macro_f1_P']:.3f} "
          f"| micro-F1(P) {s['pooled_micro_f1_P']:.3f} "
          f"| weighted-F1(P) {s['pooled_weighted_f1_P']:.3f} "
          f"| accuracy {s['pooled_accuracy']:.3f} "
          f"| alpha {s['pooled_alpha']:.3f}")
    print("     pooled family F1(P): " + " | ".join(
        f"{family}={s[f'pooled_f1_{family}']:.3f}"
        for family in scoring.POOLED_FAMILIES
    ) + "\n")

results = pd.DataFrame(test_rows)
summary_cols = ['prompt', 'valid_rate', 'macro_f1_P', 'micro_f1_P',
                'weighted_f1_P', 'accuracy', 'alpha',
                *family_f1_cols, 'latency_s']
pooled_summary_cols = ['prompt', 'valid_rate', 'pooled_macro_f1_P',
                       'pooled_micro_f1_P', 'pooled_weighted_f1_P',
                       'pooled_accuracy', 'pooled_alpha',
                       *pooled_family_f1_cols, 'latency_s']
summary = results[summary_cols].round(3)
pooled_summary = results[pooled_summary_cols].round(3)
print('Original five-family scores')
print(summary.to_string(index=False))
print('\nPooled conceptual/procedural scores')
print(pooled_summary.to_string(index=False))
